# Part 3: Content-Based Recommendation System
This notebook uses TF-IDF vectorization and Cosine Similarity to find matching titles based on unstructured descriptive attributes.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("netflix_featured.csv")

In [2]:
# Combine metadata features into one field
df["combined_features"] = (
    df["title"].fillna("") + " " +
    df["director"].fillna("") + " " +
    df["cast"].fillna("") + " " +
    df["country"].fillna("") + " " +
    df["rating"].fillna("") + " " +
    df["listed_in"].fillna("")
)

In [ ]:
# TF-IDF Transformation
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(df["combined_features"].fillna(""))

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df.index, index=df["title"]).drop_duplicates()#store movies as indices

In [ ]:
def recommend_titles(title, cosine_sim=cosine_sim, df=df, indices=indices, top_n=10):
    if title not in indices:
        return pd.DataFrame({"message": ["Title not found"]})
    idx = indices[title]
    if isinstance(idx, (pd.Series, np.ndarray)):
        idx = idx[0]
    sim_scores = list(enumerate(cosine_sim[idx]))#enumerate() adds the movie index
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]#Sort by similarity and keep the top movies
    movie_indices = [i[0] for i in sim_scores]
    return df.iloc[movie_indices][["title", "type", "director", "listed_in", "rating", "release_year"]]#If:movie_indices = [5, 12, 7],Then:df.iloc[movie_indices]#eturns rows 5, 12, and 7 from the DataFrame.

if len(df) > 0:
    print(recommend_titles(df["title"].iloc[0], top_n=5)) 

                                         title     type           director  \
5894                Anjelah Johnson: Not Fancy    Movie          Jay Karas   
854               Creating an Army of the Dead    Movie            Unknown   
6553                                  Daffedar    Movie  Johnson Esthappan   
6398  Cabins in the Wild with Dick Strawbridge  TV Show            Unknown   
4753                         The Bleeding Edge    Movie         Kirby Dick   

                                              listed_in rating  release_year  
5894                                    Stand-Up Comedy  TV-14          2015  
854                                       Documentaries  TV-MA          2021  
6553                       Dramas, International Movies  TV-14          2017  
6398  British TV Shows, International TV Shows, Real...  TV-PG          2017  
4753                                      Documentaries  TV-14          2018  
